In [5]:
import os

In [6]:

def filter_lc3_structures(pdb_folder):
    """
    Filter only actual LC3 structures from the dataset
    """
    lc3_files = []
    non_lc3_files = []
    
    for filename in os.listdir(pdb_folder):
        if filename.endswith('.pdb'):
            file_path = os.path.join(pdb_folder, filename)
            
            # Read file to check if it's actually LC3
            with open(file_path, 'r') as f:
                content = f.read()
                
            # Check for LC3 indicators in file content
            is_lc3 = any(keyword in content.upper() for keyword in [
                'MICROTUBULE-ASSOCIATED PROTEINS 1A/1B LIGHT CHAIN 3',
                'MAP1LC3', 
                'LC3A',
                'LC3B', 
                'LC3C',
                'AUTOPHAGY-RELATED PROTEIN LC3'
            ])
            
            if is_lc3:
                lc3_files.append(filename)
            else:
                non_lc3_files.append(filename)
    
    return lc3_files, non_lc3_files

In [7]:
# Run filtering
pdb_folder = r"C:\Users\Swati_Sharma\Downloads\LC3\PDB_extracted"
lc3_files, non_lc3_files = filter_lc3_structures(pdb_folder)

print(f" Actual LC3 structures: {len(lc3_files)}")
print(f" Non-LC3 files: {len(non_lc3_files)}")

print(f"\n LC3 Files:")
for file in lc3_files[:10]:  # First 10
    print(f"   {file}")

print(f"\n Non-LC3 Files (to remove):")
for file in non_lc3_files[:5]:
    print(f"   {file}")

 Actual LC3 structures: 77
 Non-LC3 files: 17

 LC3 Files:
   1ugm.pdb
   2k6q.pdb
   2lue.pdb
   2n9x.pdb
   2ncn.pdb
   2z0d.pdb
   2z0e.pdb
   2zjd.pdb
   2zzp.pdb
   3eci.pdb

 Non-LC3 Files (to remove):
   1e0g.pdb
   2l8j.pdb
   2p82.pdb
   2zpn.pdb
   3tlv.pdb


Seperating LC3 Wildtype

In [8]:
import os
import shutil

In [2]:
print("=" * 70)
print(" LC3 FILTERING - ONLY LC3 & MUTATIONS")
print("=" * 70)

def advanced_lc3_filter(pdb_folder):
    """
    filtering to get ONLY LC3 proteins and their mutations
    """
    lc3_files = []
    rejected_files = []
    detailed_info = []
    
    # LC3 specific identifiers
    lc3_identifiers = [
        'MICROTUBULE-ASSOCIATED PROTEINS 1A/1B LIGHT CHAIN 3',
        'MAP1LC3A', 'MAP1LC3B', 'MAP1LC3C',
        'LC3A', 'LC3B', 'LC3C',
        'AUTOPHAGY-RELATED PROTEIN LC3'
    ]
    
    # Common mutation indicators
    mutation_indicators = ['MUTANT', 'MUTATION', 'G120', 'K49', 'R68', 'F52', 'W6']
    
    for filename in os.listdir(pdb_folder):
        if filename.endswith('.pdb'):
            file_path = os.path.join(pdb_folder, filename)
            
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read().upper()
                
                # Check if it's actually LC3 protein
                is_lc3 = any(identifier in content for identifier in lc3_identifiers)
                
                if is_lc3:
                    # Extract protein info
                    lines = content.split('\n')
                    header = ""
                    compnd = ""
                    
                    for line in lines[:50]:  # Check first 50 lines
                        if line.startswith('HEADER'):
                            header = line.strip()
                        if line.startswith('COMPND') and 'MOLECULE:' in line:
                            compnd = line.strip()
                    
                    # Check if it's a mutation
                    is_mutant = any(mut in content for mut in mutation_indicators)
                    mutant_type = "MUTANT" if is_mutant else "WILD-TYPE"
                    
                    file_info = {
                        'filename': filename,
                        'header': header[:100] + "..." if len(header) > 100 else header,
                        'molecule_info': compnd[:100] + "..." if len(compnd) > 100 else compnd,
                        'type': mutant_type,
                        'is_mutant': is_mutant
                    }
                    
                    detailed_info.append(file_info)
                    lc3_files.append(filename)
                    
                    print(f" {filename} - {mutant_type}")
                    if compnd:
                        print(f"    {compnd[:80]}...")
                    
                else:
                    rejected_files.append(filename)
                    print(f" {filename} - NOT LC3")
                    
            except Exception as e:
                print(f"  Error reading {filename}: {e}")
                rejected_files.append(filename)
    
    return lc3_files, rejected_files, detailed_info

 LC3 FILTERING - ONLY LC3 & MUTATIONS


In [ ]:
# Run filtering
pdb_folder = r"C:\Users\Swati_Sharma\Downloads\LC3\PDB_extracted"
target_folder = r"C:\Users\Swati_Sharma\Downloads\LC3\PDB_extracted_true"

print(" Filtering files...")
lc3_files, rejected_files, detailed_info = advanced_lc3_filter(pdb_folder)

print(f"\n" + "="*50)
print(" FILTERING RESULTS:")
print("="*50)

print(f" LC3 Files Found: {len(lc3_files)}")
print(f" Non-LC3 Files: {len(rejected_files)}")

# Count mutants vs wild-type
mutants = [info for info in detailed_info if info['is_mutant']]
wild_type = [info for info in detailed_info if not info['is_mutant']]

print(f" Mutants: {len(mutants)}")
print(f" Wild-type: {len(wild_type)}")

print(f"\n LC3 FILES DETAIL:")
for info in detailed_info[:10]:  # Show first 10
    symbol = "🧬" if info['is_mutant'] else "🐭"
    print(f"   {symbol} {info['filename']} - {info['type']}")

print(f"\n REJECTED FILES (first 10):")
for file in rejected_files[:10]:
    print(f"    {file}")

# Create clean target folder
print(f"\n CREATING CLEAN DATASET...")
if os.path.exists(target_folder):
    shutil.rmtree(target_folder)
os.makedirs(target_folder)

# Copy only LC3 files
copied_count = 0
for file in lc3_files:
    src = os.path.join(pdb_folder, file)
    dst = os.path.join(target_folder, file)
    shutil.copy2(src, dst)
    copied_count += 1

print(f" COPIED {copied_count} LC3 files to: {target_folder}")

# Save detailed info as CSV
import pandas as pd
df = pd.DataFrame(detailed_info)
csv_path = os.path.join(target_folder, "lc3_dataset_info.csv")
df.to_csv(csv_path, index=False)
print(f" Saved dataset info to: {csv_path}")

print(f"\n FINAL DATASET READY FOR ESM-2 EMBEDDINGS!")
print(f"    Location: {target_folder}")
print(f"    Total LC3 structures: {len(lc3_files)}")
print(f"    Mutants: {len(mutants)}")
print(f"    Wild-type: {len(wild_type)}")

 Filtering files...
 1e0g.pdb - NOT LC3
 1ugm.pdb - WILD-TYPE
    COMPND   2 MOLECULE: MICROTUBULE-ASSOCIATED PROTEINS 1A/1B LIGHT CHAIN 3;...
 2k6q.pdb - WILD-TYPE
    COMPND  10 MOLECULE: P62_PEPTIDE FROM SEQUESTOSOME-1;...
 2l8j.pdb - NOT LC3
 2lue.pdb - WILD-TYPE
    COMPND  11 MOLECULE: OPTINEURIN;...
 2n9x.pdb - WILD-TYPE
    COMPND  10 MOLECULE: FUN14 DOMAIN-CONTAINING PROTEIN 1;...
 2ncn.pdb - WILD-TYPE
    COMPND   2 MOLECULE: AUTOPHAGY-RELATED PROTEIN LC3C;...
 2p82.pdb - NOT LC3
 2z0d.pdb - MUTANT
    COMPND  12 MOLECULE: MICROTUBULE-ASSOCIATED PROTEINS 1A/1B LIGHT CHAIN 3B;...
 2z0e.pdb - MUTANT
    COMPND  12 MOLECULE: MICROTUBULE-ASSOCIATED PROTEINS 1A/1B LIGHT CHAIN 3B;...
 2zjd.pdb - WILD-TYPE
    COMPND  11 MOLECULE: UNDECAMERIC PEPTIDE FROM SEQUESTOSOME-1;...
 2zpn.pdb - NOT LC3
 2zzp.pdb - MUTANT
    COMPND  12 MOLECULE: MICROTUBULE-ASSOCIATED PROTEINS 1A/1B LIGHT CHAIN 3B;...
 3eci.pdb - WILD-TYPE
    COMPND   2 MOLECULE: MICROTUBULE-ASSOCIATED PROTEIN 1 LIGHT CHAIN

Seperating the LC3 wildtype

In [11]:
import os
import shutil
import pandas as pd

In [12]:
print("=" * 70)
print(" ADVANCED LC3 FILTERING - WILD-TYPE ONLY")
print("=" * 70)

def advanced_lc3_filter(pdb_folder):
    """
    Advanced filtering to get ONLY wild-type LC3 proteins
    """
    wildtype_files = []
    rejected_files = []
    detailed_info = []
    
    # LC3 specific identifiers
    lc3_identifiers = [
        'MICROTUBULE-ASSOCIATED PROTEINS 1A/1B LIGHT CHAIN 3',
        'MAP1LC3A', 'MAP1LC3B', 'MAP1LC3C',
        'LC3A', 'LC3B', 'LC3C',
        'AUTOPHAGY-RELATED PROTEIN LC3'
    ]
    
    # Common mutation indicators
    mutation_indicators = ['MUTANT', 'MUTATION', 'G120', 'K49', 'R68', 'F52', 'W6']
    
    for filename in os.listdir(pdb_folder):
        if filename.endswith('.pdb'):
            file_path = os.path.join(pdb_folder, filename)
            
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read().upper()
                
                # Check if it's actually LC3 protein
                is_lc3 = any(identifier in content for identifier in lc3_identifiers)
                
                if is_lc3:
                    # Check if it's a mutation
                    is_mutant = any(mut in content for mut in mutation_indicators)
                    
                    # ONLY keep wild-type files
                    if not is_mutant:
                        # Extract protein info
                        lines = content.split('\n')
                        header = ""
                        compnd = ""
                        
                        for line in lines[:50]:
                            if line.startswith('HEADER'):
                                header = line.strip()
                            if line.startswith('COMPND') and 'MOLECULE:' in line:
                                compnd = line.strip()
                        
                        file_info = {
                            'filename': filename,
                            'header': header[:100] + "..." if len(header) > 100 else header,
                            'molecule_info': compnd[:100] + "..." if len(compnd) > 100 else compnd,
                            'type': 'WILD-TYPE'
                        }
                        
                        detailed_info.append(file_info)
                        wildtype_files.append(filename)
                        print(f" [KEPT] {filename} - WILD-TYPE")
                        if compnd:
                            print(f"    {compnd[:80]}...")
                    else:
                        rejected_files.append(filename)
                        print(f" [REJECTED] {filename} - MUTANT")
                else:
                    rejected_files.append(filename)
                    print(f" [REJECTED] {filename} - NOT LC3")
                    
            except Exception as e:
                print(f"  Error reading {filename}: {e}")
                rejected_files.append(filename)
    
    return wildtype_files, rejected_files, detailed_info


 ADVANCED LC3 FILTERING - WILD-TYPE ONLY


In [13]:
# Run advanced filtering
pdb_folder = r"C:\Users\Swati_Sharma\Downloads\LC3\PDB_extracted"
wildtype_folder = r"C:\Users\Swati_Sharma\Downloads\LC3\wildtype_dataset"

print(" Filtering files...")
wildtype_files, rejected_files, detailed_info = advanced_lc3_filter(pdb_folder)

print(f"\n" + "="*50)
print(" FILTERING RESULTS:")
print("="*50)
print(f" Wild-type LC3 Files: {len(wildtype_files)}")
print(f" Rejected Files: {len(rejected_files)}")

# Create wild-type only folder
print(f"\n CREATING WILD-TYPE DATASET...")
if os.path.exists(wildtype_folder):
    shutil.rmtree(wildtype_folder)
os.makedirs(wildtype_folder)

# Copy only wild-type files
copied_count = 0
for file in wildtype_files:
    src = os.path.join(pdb_folder, file)
    dst = os.path.join(wildtype_folder, file)
    shutil.copy2(src, dst)
    copied_count += 1

# Save wild-type info as CSV
df = pd.DataFrame(detailed_info)
csv_path = os.path.join(wildtype_folder, "wildtype_lc3_info.csv")
df.to_csv(csv_path, index=False)

print(f"\n WILD-TYPE DATASET CREATED!")
print(f"    Location: {wildtype_folder}")
print(f"    Wild-type LC3 structures: {len(wildtype_files)}")
print(f"    Dataset info saved to: {csv_path}")

 Filtering files...
 [REJECTED] 1e0g.pdb - NOT LC3
 [KEPT] 1ugm.pdb - WILD-TYPE
    COMPND   2 MOLECULE: MICROTUBULE-ASSOCIATED PROTEINS 1A/1B LIGHT CHAIN 3;...
 [KEPT] 2k6q.pdb - WILD-TYPE
    COMPND  10 MOLECULE: P62_PEPTIDE FROM SEQUESTOSOME-1;...
 [REJECTED] 2l8j.pdb - NOT LC3
 [KEPT] 2lue.pdb - WILD-TYPE
    COMPND  11 MOLECULE: OPTINEURIN;...
 [KEPT] 2n9x.pdb - WILD-TYPE
    COMPND  10 MOLECULE: FUN14 DOMAIN-CONTAINING PROTEIN 1;...
 [KEPT] 2ncn.pdb - WILD-TYPE
    COMPND   2 MOLECULE: AUTOPHAGY-RELATED PROTEIN LC3C;...
 [REJECTED] 2p82.pdb - NOT LC3
 [REJECTED] 2z0d.pdb - MUTANT
 [REJECTED] 2z0e.pdb - MUTANT
 [KEPT] 2zjd.pdb - WILD-TYPE
    COMPND  11 MOLECULE: UNDECAMERIC PEPTIDE FROM SEQUESTOSOME-1;...
 [REJECTED] 2zpn.pdb - NOT LC3
 [REJECTED] 2zzp.pdb - MUTANT
 [KEPT] 3eci.pdb - WILD-TYPE
    COMPND   2 MOLECULE: MICROTUBULE-ASSOCIATED PROTEIN 1 LIGHT CHAIN 3 ALPHA;...
 [REJECTED] 3tlv.pdb - NOT LC3
 [KEPT] 3vtu.pdb - WILD-TYPE
    COMPND   2 MOLECULE: MICROTUBULE-ASSOCIATED

Analysis of wildtype files

In [14]:
import os
import pandas as pd
from collections import Counter

print("=" * 70)
print(" WILD-TYPE LC3 PDB ANALYSIS")
print("=" * 70)

def analyze_pdb_files(wildtype_folder):
    """
    Analyze wild-type PDB files for structure, function, and sequence information
    """
    analysis_results = []
    
    for filename in os.listdir(wildtype_folder):
        if filename.endswith('.pdb'):
            file_path = os.path.join(wildtype_folder, filename)
            
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                
                lines = content.split('\n')
                
                # Initialize analysis dictionary
                file_analysis = {
                    'filename': filename,
                    'has_structure': False,
                    'has_sequence': False,
                    'has_function': False,
                    'resolution': 'N/A',
                    'experiment_type': 'N/A',
                    'sequence_length': 0,
                    'chains': [],
                    'function_info': [],
                    'missing_residues': False
                }
                
                # Parse PDB file
                for line in lines:
                    # Structural information
                    if line.startswith('EXPDTA'):
                        file_analysis['experiment_type'] = line[10:].strip()
                        file_analysis['has_structure'] = True
                    elif line.startswith('REMARK   2 RESOLUTION.'):
                        # Extract resolution and clean it
                        resolution_text = line[23:30].strip()
                        # Handle non-numeric resolution values
                        if resolution_text and resolution_text not in ['NOT APP', 'NULL', 'N/A']:
                            try:
                                file_analysis['resolution'] = float(resolution_text)
                            except ValueError:
                                file_analysis['resolution'] = resolution_text
                        else:
                            file_analysis['resolution'] = resolution_text
                        file_analysis['has_structure'] = True
                    elif line.startswith('HEADER'):
                        file_analysis['has_structure'] = True
                    
                    # Functional information
                    if line.startswith('COMPND') and 'MOLECULE:' in line:
                        file_analysis['function_info'].append(line.strip())
                        file_analysis['has_function'] = True
                    elif line.startswith('SOURCE') or line.startswith('KEYWDS'):
                        file_analysis['function_info'].append(line.strip())
                    
                    # Sequence information
                    if line.startswith('SEQRES'):
                        file_analysis['has_sequence'] = True
                        chain = line[11:12].strip()
                        if chain and chain not in file_analysis['chains']:
                            file_analysis['chains'].append(chain)
                    
                    # Check for missing residues
                    if 'MISSING' in line and 'RESIDUES' in line:
                        file_analysis['missing_residues'] = True
                
                # Count atoms and residues for structural completeness
                atom_lines = [l for l in lines if l.startswith('ATOM')]
                hetatm_lines = [l for l in lines if l.startswith('HETATM')]
                
                if atom_lines:
                    file_analysis['has_structure'] = True
                    file_analysis['atom_count'] = len(atom_lines)
                    file_analysis['hetatm_count'] = len(hetatm_lines)
                    
                    # Estimate sequence length from ATOM records
                    residues = set()
                    for line in atom_lines:
                        if len(line) >= 26:
                            res_num = line[22:26].strip()
                            chain_id = line[21:22].strip()
                            residues.add(f"{chain_id}_{res_num}")
                    file_analysis['sequence_length'] = len(residues)
                
                analysis_results.append(file_analysis)
                print(f" Analyzed: {filename}")
                
            except Exception as e:
                print(f" Error analyzing {filename}: {e}")
    
    return analysis_results

def generate_summary_report(analysis_results):
    """
    Generate a comprehensive summary report
    """
    print(f"\n" + "="*70)
    print(" ANALYSIS SUMMARY REPORT")
    print("="*70)
    
    total_files = len(analysis_results)
    
    # Basic statistics
    has_structure = sum(1 for r in analysis_results if r['has_structure'])
    has_sequence = sum(1 for r in analysis_results if r['has_sequence'])
    has_function = sum(1 for r in analysis_results if r['has_function'])
    
    print(f"\nDATASET OVERVIEW:")
    print(f" Total wild-type PDB files: {total_files}")
    print(f" Files with structural data: {has_structure} ({has_structure/total_files*100:.1f}%)")
    print(f" Files with sequence data: {has_sequence} ({has_sequence/total_files*100:.1f}%)")
    print(f" Files with functional annotations: {has_function} ({has_function/total_files*100:.1f}%)")
    
    # Structure quality analysis - handle non-numeric resolutions
    resolutions = []
    for r in analysis_results:
        resolution = r['resolution']
        if isinstance(resolution, (int, float)):
            resolutions.append(resolution)
        elif resolution != 'N/A' and resolution not in ['NOT APP', 'NULL']:
            try:
                resolutions.append(float(resolution))
            except (ValueError, TypeError):
                pass
    
    if resolutions:
        print(f"\nSTRUCTURAL QUALITY:")
        print(f" Average resolution: {sum(resolutions)/len(resolutions):.2f} Å")
        print(f" Best resolution: {min(resolutions):.2f} Å")
        print(f" Worst resolution: {max(resolutions):.2f} Å")
        print(f" Structures with numeric resolution: {len(resolutions)}/{total_files}")
    else:
        print(f"\nSTRUCTURAL QUALITY: No numeric resolution data available")
    
    # Experiment methods
    exp_methods = [r['experiment_type'] for r in analysis_results if r['experiment_type'] != 'N/A']
    if exp_methods:
        method_counts = Counter(exp_methods)
        print(f"\nEXPERIMENTAL METHODS:")
        for method, count in method_counts.most_common():
            print(f"  {method}: {count} files")
    
    # Sequence information
    seq_lengths = [r['sequence_length'] for r in analysis_results if r['sequence_length'] > 0]
    if seq_lengths:
        print(f"\nSEQUENCE INFORMATION:")
        print(f" Average sequence length: {sum(seq_lengths)/len(seq_lengths):.1f} residues")
        print(f" Shortest sequence: {min(seq_lengths)} residues")
        print(f" Longest sequence: {max(seq_lengths)} residues")
    
    # Chain information
    all_chains = []
    for r in analysis_results:
        all_chains.extend(r['chains'])
    if all_chains:
        chain_counts = Counter(all_chains)
        print(f"\nCHAIN DISTRIBUTION:")
        for chain, count in chain_counts.most_common():
            print(f"  Chain {chain}: {count} files")
    
    # Missing residues
    missing_res = sum(1 for r in analysis_results if r['missing_residues'])
    print(f"\nDATA COMPLETENESS:")
    print(f" Files with missing residues: {missing_res} ({missing_res/total_files*100:.1f}%)")
    
    # Show resolution types
    resolution_types = Counter()
    for r in analysis_results:
        resolution = r['resolution']
        if isinstance(resolution, (int, float)):
            resolution_types['Numeric'] += 1
        elif resolution == 'NOT APP':
            resolution_types['Not Applicable'] += 1
        elif resolution == 'N/A':
            resolution_types['Not Available'] += 1
        else:
            resolution_types['Other'] += 1
    
    print(f"\nRESOLUTION DATA TYPES:")
    for res_type, count in resolution_types.most_common():
        print(f"  {res_type}: {count} files")
    
    # File-by-file details
    print(f"\nDETAILED FILE ANALYSIS (first 15 files):")
    print("-" * 90)
    print(f"{'Filename':<12} {'Structure':<10} {'Sequence':<10} {'Function':<10} {'Resolution':<12} {'Length':<8} {'Method':<15}")
    print("-" * 90)
    
    for result in analysis_results[:15]:
        structure = "YES" if result['has_structure'] else "NO"
        sequence = "YES" if result['has_sequence'] else "NO"
        function = "YES" if result['has_function'] else "NO"
        resolution = str(result['resolution']) if result['resolution'] != 'N/A' else "N/A"
        length = result['sequence_length'] if result['sequence_length'] > 0 else "N/A"
        method = result['experiment_type'][:14] if result['experiment_type'] != 'N/A' else "N/A"
        
        print(f"{result['filename']:<12} {structure:<10} {sequence:<10} {function:<10} {resolution:<12} {length:<8} {method:<15}")
    
    if total_files > 15:
        print(f"... and {total_files - 15} more files")

# Run analysis
wildtype_folder = r"C:\Users\Swati_Sharma\Downloads\LC3\wildtype_dataset"

print("Analyzing wild-type PDB files...")
analysis_results = analyze_pdb_files(wildtype_folder)

if analysis_results:
    generate_summary_report(analysis_results)
    
    # Save detailed analysis to CSV
    df = pd.DataFrame(analysis_results)
    csv_path = os.path.join(wildtype_folder, "wildtype_analysis_detailed.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nDetailed analysis saved to: {csv_path}")
else:
    print("No PDB files found or analysis failed.")

 WILD-TYPE LC3 PDB ANALYSIS
Analyzing wild-type PDB files...
 Analyzed: 1ugm.pdb
 Analyzed: 2k6q.pdb
 Analyzed: 2lue.pdb
 Analyzed: 2n9x.pdb
 Analyzed: 2ncn.pdb
 Analyzed: 2zjd.pdb
 Analyzed: 3eci.pdb
 Analyzed: 3vtu.pdb
 Analyzed: 3vvw.pdb
 Analyzed: 3wal.pdb
 Analyzed: 3wam.pdb
 Analyzed: 3wan.pdb
 Analyzed: 3wao.pdb
 Analyzed: 3wap.pdb
 Analyzed: 3x0w.pdb
 Analyzed: 4waa.pdb
 Analyzed: 4zdv.pdb
 Analyzed: 5cx3.pdb
 Analyzed: 5d94.pdb
 Analyzed: 5dcn.pdb
 Analyzed: 5dpr.pdb
 Analyzed: 5dpw.pdb
 Analyzed: 5gmv.pdb
 Analyzed: 5ms2.pdb
 Analyzed: 5ms5.pdb
 Analyzed: 5ms6.pdb
 Analyzed: 5v4k.pdb
 Analyzed: 5wrd.pdb
 Analyzed: 5xac.pdb
 Analyzed: 5xad.pdb
 Analyzed: 5xae.pdb
 Analyzed: 5yiq.pdb
 Analyzed: 5yis.pdb
 Analyzed: 6lan.pdb
 Analyzed: 6tbe.pdb
 Analyzed: 7elg.pdb
 Analyzed: 7r9w.pdb
 Analyzed: 7r9z.pdb
 Analyzed: 7ra0.pdb
 Analyzed: 8q7k.pdb
 Analyzed: 8t2l.pdb
 Analyzed: 8t2m.pdb
 Analyzed: 8t2n.pdb
 Analyzed: 8t35.pdb
 Analyzed: 8t4t.pdb
 Analyzed: 8yv6.pdb

 ANALYSIS SUMMARY 

Bidirectional ESM3 for LC3 protein

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import warnings
from Bio.PDB import PDBParser, PPBuilder
import glob

In [24]:
warnings.filterwarnings("ignore")

In [25]:
print("=" * 70)
print("ESM3 FOR LC3 PROTEINS")
print("=" * 70)

ESM3 FOR LC3 PROTEINS


In [27]:
class FixedVectorQuantizer(nn.Module):
    """Simplified VQ with proper gradient flow"""
    
    def __init__(self, num_tokens=512, hidden_dim=256, commitment_cost=0.25):
        super().__init__()
        self.num_tokens = num_tokens
        self.hidden_dim = hidden_dim
        self.commitment_cost = commitment_cost
        
        self.codebook = nn.Embedding(num_tokens, hidden_dim)
        nn.init.uniform_(self.codebook.weight, -1.0, 1.0)
        
    def forward(self, inputs):
        batch_size, seq_len, hidden_dim = inputs.shape
        flat_inputs = inputs.reshape(-1, hidden_dim)
        
        distances = torch.cdist(flat_inputs, self.codebook.weight)
        encoding_indices = torch.argmin(distances, dim=1)
        quantized = self.codebook(encoding_indices).view(batch_size, seq_len, hidden_dim)
        
        quantized_st = inputs + (quantized - inputs).detach()
        
        commitment_loss = F.mse_loss(inputs, quantized.detach())
        codebook_loss = F.mse_loss(quantized, inputs.detach())
        vq_loss = codebook_loss + self.commitment_cost * commitment_loss
        
        return quantized_st, encoding_indices.view(batch_size, seq_len), vq_loss


In [28]:
class LC3FunctionTokenizer(nn.Module):
    """LC3-specific function tokenizer"""
    
    def __init__(self, vocab_size=16, hidden_dim=256):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        
        self.function_categories = {
            'n_terminal': 0, 'c_terminal': 1, 'lir_binding': 2,
            'ubiquitin_fold': 3, 'hydrophobic_core': 4, 'surface_exposed': 5,
            'membrane_binding': 6, 'autophagy_core': 7, 'conserved': 8,
            'structural': 9, 'catalytic': 10, 'binding_site': 11,
            'unknown': 12
        }
        
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        
    def predict_lc3_functions(self, sequence, coords):
        batch_size, seq_len = sequence.shape[0], sequence.shape[1]
        function_tokens = torch.zeros(batch_size, seq_len, dtype=torch.long)
        
        for i in range(batch_size):
            seq = sequence[i]
            for j in range(seq_len):
                if j < 10:
                    function_tokens[i, j] = self.function_categories['n_terminal']
                elif j > seq_len - 10:
                    function_tokens[i, j] = self.function_categories['c_terminal']
                elif (44 <= j <= 52) or (49 <= j <= 56):
                    function_tokens[i, j] = self.function_categories['lir_binding']
                elif 10 <= j <= 110:
                    function_tokens[i, j] = self.function_categories['ubiquitin_fold']
                elif (23 <= j <= 28) or (35 <= j <= 40) or (65 <= j <= 70):
                    function_tokens[i, j] = self.function_categories['hydrophobic_core']
                else:
                    function_tokens[i, j] = self.function_categories['surface_exposed']
        
        return function_tokens
    
    def forward(self, sequence_tokens, coords):
        return self.predict_lc3_functions(sequence_tokens, coords)


In [29]:
class BidirectionalESM3(nn.Module):
    """ Bidirectional encoder like ESM3 for representation learning"""
    
    def __init__(self, vocab_size=33, struct_tokens=512, func_tokens=16,
                 hidden_dim=256, num_layers=6, num_heads=8, max_length=2048):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.max_length = max_length
        
        # Embedding layers
        self.seq_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.struct_embedding = nn.Embedding(struct_tokens, hidden_dim)
        self.func_embedding = nn.Embedding(func_tokens, hidden_dim)
        self.modality_embeddings = nn.Embedding(3, hidden_dim)
        
        # Positional encoding
        self.pos_encoding = nn.Parameter(torch.randn(1, max_length, hidden_dim))
        
        #  Use TransformerEncoder for bidirectional context
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Output heads
        self.seq_head = nn.Linear(hidden_dim, vocab_size)
        self.struct_head = nn.Linear(hidden_dim, struct_tokens)
        self.func_head = nn.Linear(hidden_dim, func_tokens)
        
    def interleave_modalities(self, seq_tokens, struct_tokens, func_tokens, device):
        """Interleave S→St→F stream with modality embeddings"""
        batch_size, seq_len = seq_tokens.shape
        
        # Embed each modality
        seq_emb = self.seq_embedding(seq_tokens)
        struct_emb = self.struct_embedding(struct_tokens)
        func_emb = self.func_embedding(func_tokens)
        
        # Interleave: [seq1, struct1, func1, seq2, struct2, func2, ...]
        interleaved = torch.stack([seq_emb, struct_emb, func_emb], dim=2)
        interleaved = interleaved.view(batch_size, seq_len * 3, self.hidden_dim)
        
        # Add modality type embeddings
        modality_ids = torch.tensor([0, 1, 2] * seq_len, device=device).unsqueeze(0)
        modality_emb = self.modality_embeddings(modality_ids)
        interleaved = interleaved + modality_emb
        
        # Add positional encoding
        total_tokens = seq_len * 3
        if total_tokens > self.max_length:
            total_tokens = self.max_length
            interleaved = interleaved[:, :self.max_length, :]
        
        pos_emb = self.pos_encoding[:, :total_tokens, :]
        interleaved = interleaved + pos_emb
        
        return interleaved
    
    def forward(self, seq_tokens, struct_tokens, func_tokens):
        """Bidirectional forward pass - NO CAUSAL MASKING"""
        batch_size, seq_len = seq_tokens.shape
        
        # Interleave all modalities
        interleaved = self.interleave_modalities(seq_tokens, struct_tokens, func_tokens, seq_tokens.device)
        
        # Bidirectional transformer (no causal mask)
        encoded = self.transformer(interleaved)
        
        #  Proper deinterleaving without padding corruption
        batch_size, total_len, hidden_dim = encoded.shape
        actual_seq_len = total_len // 3
        
        # Handle edge case where we have partial sequences
        if total_len % 3 != 0:
            # Use only the complete residues
            complete_len = (total_len // 3) * 3
            encoded = encoded[:, :complete_len, :]
            total_len = complete_len
            actual_seq_len = total_len // 3
        
        # Reshape to [batch, actual_seq_len, 3, hidden]
        deinterleaved = encoded.view(batch_size, actual_seq_len, 3, hidden_dim)
        
        # Split modalities - CORRECTED: slice to match original sequence length
        seq_encoded = deinterleaved[:, :min(actual_seq_len, seq_len), 0, :]
        struct_encoded = deinterleaved[:, :min(actual_seq_len, seq_len), 1, :]
        func_encoded = deinterleaved[:, :min(actual_seq_len, seq_len), 2, :]
        
        # Output predictions
        seq_logits = self.seq_head(seq_encoded)
        struct_logits = self.struct_head(struct_encoded)
        func_logits = self.func_head(func_encoded)
        
        return seq_logits, struct_logits, func_logits, encoded


In [ ]:
class WorkingLC3ESM3:
    """Bidirectional MLM with PROPER VQ loss"""
    
    def __init__(self, hidden_dim=256, max_sequence_length=500):
        self.structure_tokenizer = FixedStructureTokenizer()
        self.function_tokenizer = LC3FunctionTokenizer(vocab_size=16)
        self.model = BidirectionalESM3(
            hidden_dim=hidden_dim, 
            max_length=max_sequence_length * 3,
            func_tokens=16
        )
        self.max_sequence_length = max_sequence_length
        
        self.aa_to_token = {aa: i for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}
        self.token_to_aa = {i: aa for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}
        
    def extract_coordinates_single_chain(self, structure):
        """Extract coordinates from FIRST chain only to match sequence"""
        coords = []
        for model in structure:
            for chain in model:
                chain_coords = []
                for residue in chain:
                    if 'CA' in residue:
                        ca = residue['CA']
                        chain_coords.append(ca.get_coord())
                if chain_coords:
                    return torch.tensor(chain_coords, dtype=torch.float32)
        return None
    
    def process_pdb_file(self, pdb_file):
        try:
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure('lc3', pdb_file)
            
            ppb = PPBuilder()
            sequences = []
            for pp in ppb.build_peptides(structure):
                sequences.append(str(pp.get_sequence()))
            
            if not sequences:
                raise ValueError(f"No sequence found in {pdb_file}")
                
            sequence = sequences[0]
            seq_tokens = torch.tensor([self.aa_to_token.get(aa, 0) for aa in sequence])
            
            coords = self.extract_coordinates_single_chain(structure)
            if coords is None:
                raise ValueError("No coordinates found")
            
            if len(seq_tokens) != len(coords):
                min_len = min(len(seq_tokens), len(coords))
                if abs(len(seq_tokens) - len(coords)) > 5:
                    print(f"Length adjustment: seq={len(seq_tokens)}->{min_len}, coords={len(coords)}->{min_len} for {os.path.basename(pdb_file)}")
                seq_tokens = seq_tokens[:min_len]
                coords = coords[:min_len]
            
            if len(seq_tokens) > self.max_sequence_length:
                print(f"Skipping long sequence: {len(seq_tokens)} > {self.max_sequence_length}")
                return None
            
            if len(seq_tokens) < 10:
                print(f"Skipping very short sequence: {len(seq_tokens)} residues")
                return None
            
            struct_tokens, vq_loss = self.structure_tokenizer(coords.unsqueeze(0))
            struct_tokens = struct_tokens[0]
            
            func_tokens = self.function_tokenizer(
                seq_tokens.unsqueeze(0), 
                coords.unsqueeze(0)
            )[0]
            
            assert len(seq_tokens) == len(struct_tokens) == len(func_tokens)
            
            return {
                'seq_tokens': seq_tokens,
                'struct_tokens': struct_tokens,
                'func_tokens': func_tokens,
                'sequence': sequence,
                'coords': coords,
                'pdb_id': os.path.basename(pdb_file).replace('.pdb', ''),
                'vq_loss': vq_loss.item()  # FIXED: Store as scalar value, not tensor
            }
            
        except Exception as e:
            print(f"Error processing {pdb_file}: {e}")
            return None
    
    def load_lc3_dataset(self, pdb_folder):
        print(f"Loading LC3 PDB files from {pdb_folder}")
        
        pdb_files = glob.glob(os.path.join(pdb_folder, "*.pdb"))
        if not pdb_files:
            pdb_files = glob.glob(os.path.join(pdb_folder, "*.ent"))
        
        print(f"Found {len(pdb_files)} PDB files")
        
        all_data = []
        successful_files = 0
        
        for pdb_file in pdb_files:
            result = self.process_pdb_file(pdb_file)
            if result is not None:
                all_data.append(result)
                successful_files += 1
                print(f"Processed {result['pdb_id']} - Length: {len(result['seq_tokens'])}")
        
        print(f"Successfully processed {successful_files}/{len(pdb_files)} PDB files")
        
        if successful_files == 0:
            raise ValueError("No valid PDB files could be processed.")
            
        return all_data
    
    def create_dataloader_with_vq_loss(self, dataset, batch_size=4):
        """CORRECTED: Store VQ loss as scalar values"""
        def collate_fn(batch):
            seq_tokens = [item['seq_tokens'] for item in batch]
            struct_tokens = [item['struct_tokens'] for item in batch]
            func_tokens = [item['func_tokens'] for item in batch]
            vq_losses = [item['vq_loss'] for item in batch]  # Store VQ losses as scalars
            
            max_len = max(len(tokens) for tokens in seq_tokens)
            
            padded_seqs = []
            padded_structs = []
            padded_funcs = []
            
            for seq, struct, func in zip(seq_tokens, struct_tokens, func_tokens):
                pad_len = max_len - len(seq)
                padded_seqs.append(F.pad(seq, (0, pad_len), value=0))
                padded_structs.append(F.pad(struct, (0, pad_len), value=0))
                padded_funcs.append(F.pad(func, (0, pad_len), value=0))
            
            return (
                torch.stack(padded_seqs),
                torch.stack(padded_structs),
                torch.stack(padded_funcs),
                torch.tensor(vq_losses)  # Convert to tensor
            )
        
        return torch.utils.data.DataLoader(
            dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn
        )
    
    def train_bidirectional_mlm(self, pdb_folder, num_epochs=20, batch_size=2):
        """CORRECTED: Bidirectional MLM training with PROPER VQ loss"""
        print("Starting BIDIRECTIONAL MLM ESM3 Training")
        print("Using 15% masking + PROPER VQ loss")
        
        dataset = self.load_lc3_dataset(pdb_folder)
        dataloader = self.create_dataloader_with_vq_loss(dataset, batch_size)
        
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4, weight_decay=0.01)
        criterion = nn.CrossEntropyLoss(ignore_index=0)
        
        print(f"Training on {len(dataset)} LC3 structures for {num_epochs} epochs")
        
        best_loss = float('inf')
        best_model_state = None
        
        for epoch in range(num_epochs):
            self.model.train()
            total_loss = 0
            total_seq_loss = 0
            total_struct_loss = 0
            total_func_loss = 0
            total_vq_loss = 0
            batches_processed = 0
            
            for batch_idx, (seq_tokens, struct_tokens, func_tokens, vq_losses) in enumerate(dataloader):
                if seq_tokens.shape[0] < 2:
                    continue
                
                batch_size, seq_len = seq_tokens.shape
                device = seq_tokens.device
                
                # CORRECTED: Standard 15% masking like ESM3
                mask_prob = 0.15
                masked_seq = self.mask_tokens(seq_tokens, mask_prob)
                masked_struct = self.mask_tokens(struct_tokens, mask_prob)
                masked_func = self.mask_tokens(func_tokens, mask_prob)
                
                # Forward pass with masked inputs - BIDIRECTIONAL
                seq_logits, struct_logits, func_logits, _ = self.model(
                    masked_seq, masked_struct, masked_func
                )
                
                # CORRECTED: Compute reconstruction losses
                seq_loss = criterion(seq_logits.reshape(-1, seq_logits.size(-1)), 
                                   seq_tokens.reshape(-1))
                struct_loss = criterion(struct_logits.reshape(-1, struct_logits.size(-1)), 
                                      struct_tokens.reshape(-1))
                func_loss = criterion(func_logits.reshape(-1, func_logits.size(-1)), 
                                    func_tokens.reshape(-1))
                
                # CORRECTED: Use stored VQ loss (already computed during preprocessing)
                vq_loss = vq_losses.mean()  # Average VQ loss across batch
                
                # CORRECTED: Total loss with VQ integration
                total_batch_loss = (0.6 * seq_loss + 
                                  0.3 * struct_loss + 
                                  0.1 * func_loss + 
                                  0.25 * vq_loss)  # VQ loss weight
                
                optimizer.zero_grad()
                total_batch_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                
                total_loss += total_batch_loss.item()
                total_seq_loss += seq_loss.item()
                total_struct_loss += struct_loss.item()
                total_func_loss += func_loss.item()
                total_vq_loss += vq_loss.item()
                batches_processed += 1
            
            if batches_processed == 0:
                print(f"No batches processed in epoch {epoch+1}")
                continue
                
            avg_loss = total_loss / batches_processed
            
            if avg_loss < best_loss:
                best_loss = avg_loss
                best_model_state = self.model.state_dict().copy()
            
            print(f"Epoch {epoch+1}/{num_epochs}:")
            print(f"  Total Loss: {avg_loss:.4f}")
            print(f"  Seq Loss: {total_seq_loss/batches_processed:.4f}")
            print(f"  Struct Loss: {total_struct_loss/batches_processed:.4f}")
            print(f"  Func Loss: {total_func_loss/batches_processed:.4f}")
            print(f"  VQ Loss: {total_vq_loss/batches_processed:.4f}")
            print("-" * 50)
        
        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
            print(f"Loaded best model with loss: {best_loss:.4f}")
        
        print("BIDIRECTIONAL MLM ESM3 Training Completed Successfully!")
        return dataset
    
    def mask_tokens(self, tokens, mask_prob=0.15):
        """Standard 15% masking like ESM3"""
        mask = torch.rand(tokens.shape) < mask_prob
        masked_tokens = tokens.clone()
        masked_tokens[mask] = 0  # Mask token
        return masked_tokens
    
    def get_embeddings(self, pdb_file):
        """Get meaningful bidirectional embeddings"""
        result = self.process_pdb_file(pdb_file)
        if result is None:
            return None
        
        with torch.no_grad():
            self.model.eval()
            _, _, _, embeddings = self.model(
                result['seq_tokens'].unsqueeze(0),
                result['struct_tokens'].unsqueeze(0),
                result['func_tokens'].unsqueeze(0)
            )
        
        return {
            'embeddings': embeddings.squeeze(0),
            'sequence': result['sequence'],
            'pdb_id': result['pdb_id']
        }
    
    def analyze_dataset(self, dataset):
        lengths = [len(item['seq_tokens']) for item in dataset]
        print(f"Dataset Analysis:")
        print(f"  Total structures: {len(dataset)}")
        print(f"  Min length: {min(lengths)}")
        print(f"  Max length: {max(lengths)}")
        print(f"  Avg length: {np.mean(lengths):.1f}")
        print(f"  Std length: {np.std(lengths):.1f}")

    def save_all_embeddings(self, pdb_folder, output_file):
        """Save embeddings for all proteins in the dataset"""
        print(f"Saving all embeddings to {output_file}")
        
        dataset = self.load_lc3_dataset(pdb_folder)
        all_embeddings = {}
        
        for item in dataset:
            pdb_id = item['pdb_id']
            try:
                embedding_result = self.get_embeddings(os.path.join(pdb_folder, f"{pdb_id}.pdb"))
                if embedding_result is not None:
                    all_embeddings[pdb_id] = {
                        'embeddings': embedding_result['embeddings'].cpu().numpy(),
                        'sequence': embedding_result['sequence']
                    }
                    print(f"Saved embeddings for {pdb_id} - shape: {all_embeddings[pdb_id]['embeddings'].shape}")
            except Exception as e:
                print(f"Error getting embeddings for {pdb_id}: {e}")
        
        # Save embeddings
        np.savez(output_file, **all_embeddings)
        
        # Save model
        model_file = output_file.replace('.npz', '_model.pth')
        torch.save(self.model.state_dict(), model_file)
        
        print(f"Saved {len(all_embeddings)} protein embeddings to {output_file}")
        print(f"Saved model to {model_file}")

if __name__ == "__main__":
    print("BIDIRECTIONAL MLM ESM3 - Fixed VQ Loss")
    
    working_esm3 = WorkingLC3ESM3(hidden_dim=256, max_sequence_length=500)
    
    pdb_folder = r"C:\Users\Swati_Sharma\Downloads\LC3\wildtype_dataset"
    trained_dataset = working_esm3.train_bidirectional_mlm(pdb_folder, num_epochs=10, batch_size=2)
    working_esm3.analyze_dataset(trained_dataset)
    
    print("BIDIRECTIONAL MLM ESM3 Training Completed Successfully!")

    # === FINAL LINES ===
    working_esm3.save_all_embeddings(pdb_folder, "LC3_all_proteins_embeddings.npz")
    print("ALL EMBEDDINGS + MODEL SAVED. YOU ARE DONE.")

In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import warnings
from Bio.PDB import PDBParser, PPBuilder
import glob

warnings.filterwarnings("ignore")

print("=" * 70)
print("ESM3 FOR LC3 PROTEINS - FIXED VQ LOSS")
print("=" * 70)

class FixedVectorQuantizer(nn.Module):
    """Simplified VQ with proper gradient flow"""
    
    def __init__(self, num_tokens=512, hidden_dim=256, commitment_cost=0.25):
        super().__init__()
        self.num_tokens = num_tokens
        self.hidden_dim = hidden_dim
        self.commitment_cost = commitment_cost
        
        self.codebook = nn.Embedding(num_tokens, hidden_dim)
        nn.init.uniform_(self.codebook.weight, -1.0, 1.0)
        
    def forward(self, inputs):
        batch_size, seq_len, hidden_dim = inputs.shape
        flat_inputs = inputs.reshape(-1, hidden_dim)
        
        distances = torch.cdist(flat_inputs, self.codebook.weight)
        encoding_indices = torch.argmin(distances, dim=1)
        quantized = self.codebook(encoding_indices).view(batch_size, seq_len, hidden_dim)
        
        quantized_st = inputs + (quantized - inputs).detach()
        
        commitment_loss = F.mse_loss(inputs, quantized.detach())
        codebook_loss = F.mse_loss(quantized, inputs.detach())
        vq_loss = codebook_loss + self.commitment_cost * commitment_loss
        
        return quantized_st, encoding_indices.view(batch_size, seq_len), vq_loss

class FixedStructureTokenizer(nn.Module):
    """Structure tokenizer with working VQ"""
    
    def __init__(self, num_tokens=512, hidden_dim=256):
        super().__init__()
        self.num_tokens = num_tokens
        self.hidden_dim = hidden_dim
        
        self.geometric_encoder = nn.Sequential(
            nn.Linear(22, 128),
            nn.ReLU(),
            nn.Linear(128, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        self.vq = FixedVectorQuantizer(num_tokens, hidden_dim)
        
    def extract_enhanced_features(self, coords):
        batch_size, seq_len, _ = coords.shape
        enhanced_features = []
        
        for i in range(seq_len):
            indices = [max(0, i-1), i, min(seq_len-1, i+1), min(seq_len-1, i+2)]
            indices = indices[:4]
            local_coords = coords[:, indices, :]
            features = self.calculate_geometric_features(local_coords)
            enhanced_features.append(features)
        
        return torch.stack(enhanced_features, dim=1)
    
    def calculate_geometric_features(self, local_coords):
        vec1 = local_coords[:, 1] - local_coords[:, 0]
        vec2 = local_coords[:, 2] - local_coords[:, 1]
        vec3 = local_coords[:, 3] - local_coords[:, 2]
        
        dist1 = torch.norm(vec1, dim=-1, keepdim=True)
        dist2 = torch.norm(vec2, dim=-1, keepdim=True)
        dist3 = torch.norm(vec3, dim=-1, keepdim=True)
        
        cos_angle1 = F.cosine_similarity(vec1, vec2, dim=-1).unsqueeze(-1)
        cos_angle2 = F.cosine_similarity(vec2, vec3, dim=-1).unsqueeze(-1)
        cos_angle3 = F.cosine_similarity(vec1, vec3, dim=-1).unsqueeze(-1)
        
        normal1 = torch.cross(vec1, vec2, dim=-1)
        normal2 = torch.cross(vec2, vec3, dim=-1)
        cos_dihedral = F.cosine_similarity(normal1, normal2, dim=-1).unsqueeze(-1)
        
        vec_components = torch.cat([vec1, vec2, vec3], dim=-1)
        position_features = torch.cat([local_coords[:, 1], local_coords[:, 2]], dim=-1)
        
        features = torch.cat([
            dist1, dist2, dist3, cos_angle1, cos_angle2, cos_angle3,
            cos_dihedral, vec_components, position_features
        ], dim=-1)
        
        assert features.shape[-1] == 22
        return features
    
    def forward(self, coords):
        features = self.extract_enhanced_features(coords)
        continuous_encoding = self.geometric_encoder(features)
        quantized, token_indices, vq_loss = self.vq(continuous_encoding)
        return token_indices, vq_loss

class LC3FunctionTokenizer(nn.Module):
    """LC3-specific function tokenizer"""
    
    def __init__(self, vocab_size=16, hidden_dim=256):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        
        self.function_categories = {
            'n_terminal': 0, 'c_terminal': 1, 'lir_binding': 2,
            'ubiquitin_fold': 3, 'hydrophobic_core': 4, 'surface_exposed': 5,
            'membrane_binding': 6, 'autophagy_core': 7, 'conserved': 8,
            'structural': 9, 'catalytic': 10, 'binding_site': 11,
            'unknown': 12
        }
        
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        
    def predict_lc3_functions(self, sequence, coords):
        batch_size, seq_len = sequence.shape[0], sequence.shape[1]
        function_tokens = torch.zeros(batch_size, seq_len, dtype=torch.long)
        
        for i in range(batch_size):
            seq = sequence[i]
            for j in range(seq_len):
                if j < 10:
                    function_tokens[i, j] = self.function_categories['n_terminal']
                elif j > seq_len - 10:
                    function_tokens[i, j] = self.function_categories['c_terminal']
                elif (44 <= j <= 52) or (49 <= j <= 56):
                    function_tokens[i, j] = self.function_categories['lir_binding']
                elif 10 <= j <= 110:
                    function_tokens[i, j] = self.function_categories['ubiquitin_fold']
                elif (23 <= j <= 28) or (35 <= j <= 40) or (65 <= j <= 70):
                    function_tokens[i, j] = self.function_categories['hydrophobic_core']
                else:
                    function_tokens[i, j] = self.function_categories['surface_exposed']
        
        return function_tokens
    
    def forward(self, sequence_tokens, coords):
        return self.predict_lc3_functions(sequence_tokens, coords)

class BidirectionalESM3(nn.Module):
    """CORRECTED: Bidirectional encoder like ESM3 for representation learning"""
    
    def __init__(self, vocab_size=33, struct_tokens=512, func_tokens=16,
                 hidden_dim=256, num_layers=6, num_heads=8, max_length=2048):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.max_length = max_length
        
        # Embedding layers
        self.seq_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.struct_embedding = nn.Embedding(struct_tokens, hidden_dim)
        self.func_embedding = nn.Embedding(func_tokens, hidden_dim)
        self.modality_embeddings = nn.Embedding(3, hidden_dim)
        
        # Positional encoding
        self.pos_encoding = nn.Parameter(torch.randn(1, max_length, hidden_dim))
        
        # CORRECTED: Use TransformerEncoder for bidirectional context
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Output heads
        self.seq_head = nn.Linear(hidden_dim, vocab_size)
        self.struct_head = nn.Linear(hidden_dim, struct_tokens)
        self.func_head = nn.Linear(hidden_dim, func_tokens)
        
    def interleave_modalities(self, seq_tokens, struct_tokens, func_tokens, device):
        """Interleave S→St→F stream with modality embeddings"""
        batch_size, seq_len = seq_tokens.shape
        
        # Embed each modality
        seq_emb = self.seq_embedding(seq_tokens)
        struct_emb = self.struct_embedding(struct_tokens)
        func_emb = self.func_embedding(func_tokens)
        
        # Interleave: [seq1, struct1, func1, seq2, struct2, func2, ...]
        interleaved = torch.stack([seq_emb, struct_emb, func_emb], dim=2)
        interleaved = interleaved.view(batch_size, seq_len * 3, self.hidden_dim)
        
        # Add modality type embeddings
        modality_ids = torch.tensor([0, 1, 2] * seq_len, device=device).unsqueeze(0)
        modality_emb = self.modality_embeddings(modality_ids)
        interleaved = interleaved + modality_emb
        
        # Add positional encoding
        total_tokens = seq_len * 3
        if total_tokens > self.max_length:
            total_tokens = self.max_length
            interleaved = interleaved[:, :self.max_length, :]
        
        pos_emb = self.pos_encoding[:, :total_tokens, :]
        interleaved = interleaved + pos_emb
        
        return interleaved
    
    def forward(self, seq_tokens, struct_tokens, func_tokens):
        """Bidirectional forward pass - NO CAUSAL MASKING"""
        batch_size, seq_len = seq_tokens.shape
        
        # Interleave all modalities
        interleaved = self.interleave_modalities(seq_tokens, struct_tokens, func_tokens, seq_tokens.device)
        
        # CORRECTED: Bidirectional transformer (no causal mask)
        encoded = self.transformer(interleaved)
        
        # CORRECTED: Proper deinterleaving without padding corruption
        batch_size, total_len, hidden_dim = encoded.shape
        actual_seq_len = total_len // 3
        
        # Handle edge case where we have partial sequences
        if total_len % 3 != 0:
            # Use only the complete residues
            complete_len = (total_len // 3) * 3
            encoded = encoded[:, :complete_len, :]
            total_len = complete_len
            actual_seq_len = total_len // 3
        
        # Reshape to [batch, actual_seq_len, 3, hidden]
        deinterleaved = encoded.view(batch_size, actual_seq_len, 3, hidden_dim)
        
        # Split modalities - CORRECTED: slice to match original sequence length
        seq_encoded = deinterleaved[:, :min(actual_seq_len, seq_len), 0, :]
        struct_encoded = deinterleaved[:, :min(actual_seq_len, seq_len), 1, :]
        func_encoded = deinterleaved[:, :min(actual_seq_len, seq_len), 2, :]
        
        # Output predictions
        seq_logits = self.seq_head(seq_encoded)
        struct_logits = self.struct_head(struct_encoded)
        func_logits = self.func_head(func_encoded)
        
        return seq_logits, struct_logits, func_logits, encoded

class WorkingLC3ESM3:
    """WORKING ESM3: Bidirectional MLM with PROPER VQ loss"""
    
    def __init__(self, hidden_dim=256, max_sequence_length=500):
        self.structure_tokenizer = FixedStructureTokenizer()
        self.function_tokenizer = LC3FunctionTokenizer(vocab_size=16)
        self.model = BidirectionalESM3(
            hidden_dim=hidden_dim, 
            max_length=max_sequence_length * 3,
            func_tokens=16
        )
        self.max_sequence_length = max_sequence_length
        
        self.aa_to_token = {aa: i for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}
        self.token_to_aa = {i: aa for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}
        
    def extract_coordinates_single_chain(self, structure):
        """Extract coordinates from FIRST chain only to match sequence"""
        coords = []
        for model in structure:
            for chain in model:
                chain_coords = []
                for residue in chain:
                    if 'CA' in residue:
                        ca = residue['CA']
                        chain_coords.append(ca.get_coord())
                if chain_coords:
                    return torch.tensor(chain_coords, dtype=torch.float32)
        return None
    
    def process_pdb_file(self, pdb_file):
        try:
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure('lc3', pdb_file)
            
            ppb = PPBuilder()
            sequences = []
            for pp in ppb.build_peptides(structure):
                sequences.append(str(pp.get_sequence()))
            
            if not sequences:
                raise ValueError(f"No sequence found in {pdb_file}")
                
            sequence = sequences[0]
            seq_tokens = torch.tensor([self.aa_to_token.get(aa, 0) for aa in sequence])
            
            coords = self.extract_coordinates_single_chain(structure)
            if coords is None:
                raise ValueError("No coordinates found")
            
            if len(seq_tokens) != len(coords):
                min_len = min(len(seq_tokens), len(coords))
                if abs(len(seq_tokens) - len(coords)) > 5:
                    print(f"Length adjustment: seq={len(seq_tokens)}->{min_len}, coords={len(coords)}->{min_len} for {os.path.basename(pdb_file)}")
                seq_tokens = seq_tokens[:min_len]
                coords = coords[:min_len]
            
            if len(seq_tokens) > self.max_sequence_length:
                print(f"Skipping long sequence: {len(seq_tokens)} > {self.max_sequence_length}")
                return None
            
            if len(seq_tokens) < 10:
                print(f"Skipping very short sequence: {len(seq_tokens)} residues")
                return None
            
            struct_tokens, vq_loss = self.structure_tokenizer(coords.unsqueeze(0))
            struct_tokens = struct_tokens[0]
            
            func_tokens = self.function_tokenizer(
                seq_tokens.unsqueeze(0), 
                coords.unsqueeze(0)
            )[0]
            
            assert len(seq_tokens) == len(struct_tokens) == len(func_tokens)
            
            return {
                'seq_tokens': seq_tokens,
                'struct_tokens': struct_tokens,
                'func_tokens': func_tokens,
                'sequence': sequence,
                'coords': coords,
                'pdb_id': os.path.basename(pdb_file).replace('.pdb', ''),
                'vq_loss': vq_loss.item()  # FIXED: Store as scalar value, not tensor
            }
            
        except Exception as e:
            print(f"Error processing {pdb_file}: {e}")
            return None
    
    def load_lc3_dataset(self, pdb_folder):
        print(f"Loading LC3 PDB files from {pdb_folder}")
        
        pdb_files = glob.glob(os.path.join(pdb_folder, "*.pdb"))
        if not pdb_files:
            pdb_files = glob.glob(os.path.join(pdb_folder, "*.ent"))
        
        print(f"Found {len(pdb_files)} PDB files")
        
        all_data = []
        successful_files = 0
        
        for pdb_file in pdb_files:
            result = self.process_pdb_file(pdb_file)
            if result is not None:
                all_data.append(result)
                successful_files += 1
                print(f"Processed {result['pdb_id']} - Length: {len(result['seq_tokens'])}")
        
        print(f"Successfully processed {successful_files}/{len(pdb_files)} PDB files")
        
        if successful_files == 0:
            raise ValueError("No valid PDB files could be processed.")
            
        return all_data
    
    def create_dataloader_with_vq_loss(self, dataset, batch_size=4):
        """CORRECTED: Store VQ loss as scalar values"""
        def collate_fn(batch):
            seq_tokens = [item['seq_tokens'] for item in batch]
            struct_tokens = [item['struct_tokens'] for item in batch]
            func_tokens = [item['func_tokens'] for item in batch]
            vq_losses = [item['vq_loss'] for item in batch]  # Store VQ losses as scalars
            
            max_len = max(len(tokens) for tokens in seq_tokens)
            
            padded_seqs = []
            padded_structs = []
            padded_funcs = []
            
            for seq, struct, func in zip(seq_tokens, struct_tokens, func_tokens):
                pad_len = max_len - len(seq)
                padded_seqs.append(F.pad(seq, (0, pad_len), value=0))
                padded_structs.append(F.pad(struct, (0, pad_len), value=0))
                padded_funcs.append(F.pad(func, (0, pad_len), value=0))
            
            return (
                torch.stack(padded_seqs),
                torch.stack(padded_structs),
                torch.stack(padded_funcs),
                torch.tensor(vq_losses)  # Convert to tensor
            )
        
        return torch.utils.data.DataLoader(
            dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn
        )
    
    def train_bidirectional_mlm(self, pdb_folder, num_epochs=20, batch_size=2):
        """CORRECTED: Bidirectional MLM training with PROPER VQ loss"""
        print("Starting BIDIRECTIONAL MLM ESM3 Training")
        print("Using 15% masking + PROPER VQ loss")
        
        dataset = self.load_lc3_dataset(pdb_folder)
        dataloader = self.create_dataloader_with_vq_loss(dataset, batch_size)
        
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4, weight_decay=0.01)
        criterion = nn.CrossEntropyLoss(ignore_index=0)
        
        print(f"Training on {len(dataset)} LC3 structures for {num_epochs} epochs")
        
        best_loss = float('inf')
        best_model_state = None
        
        for epoch in range(num_epochs):
            self.model.train()
            total_loss = 0
            total_seq_loss = 0
            total_struct_loss = 0
            total_func_loss = 0
            total_vq_loss = 0
            batches_processed = 0
            
            for batch_idx, (seq_tokens, struct_tokens, func_tokens, vq_losses) in enumerate(dataloader):
                if seq_tokens.shape[0] < 2:
                    continue
                
                batch_size, seq_len = seq_tokens.shape
                device = seq_tokens.device
                
                # CORRECTED: Standard 15% masking like ESM3
                mask_prob = 0.15
                masked_seq = self.mask_tokens(seq_tokens, mask_prob)
                masked_struct = self.mask_tokens(struct_tokens, mask_prob)
                masked_func = self.mask_tokens(func_tokens, mask_prob)
                
                # Forward pass with masked inputs - BIDIRECTIONAL
                seq_logits, struct_logits, func_logits, _ = self.model(
                    masked_seq, masked_struct, masked_func
                )
                
                # CORRECTED: Compute reconstruction losses
                seq_loss = criterion(seq_logits.reshape(-1, seq_logits.size(-1)), 
                                   seq_tokens.reshape(-1))
                struct_loss = criterion(struct_logits.reshape(-1, struct_logits.size(-1)), 
                                      struct_tokens.reshape(-1))
                func_loss = criterion(func_logits.reshape(-1, func_logits.size(-1)), 
                                    func_tokens.reshape(-1))
                
                # CORRECTED: Use stored VQ loss (already computed during preprocessing)
                vq_loss = vq_losses.mean()  # Average VQ loss across batch
                
                # CORRECTED: Total loss with VQ integration
                total_batch_loss = (0.6 * seq_loss + 
                                  0.3 * struct_loss + 
                                  0.1 * func_loss + 
                                  0.25 * vq_loss)  # VQ loss weight
                
                optimizer.zero_grad()
                total_batch_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                
                total_loss += total_batch_loss.item()
                total_seq_loss += seq_loss.item()
                total_struct_loss += struct_loss.item()
                total_func_loss += func_loss.item()
                total_vq_loss += vq_loss.item()
                batches_processed += 1
            
            if batches_processed == 0:
                print(f"No batches processed in epoch {epoch+1}")
                continue
                
            avg_loss = total_loss / batches_processed
            
            if avg_loss < best_loss:
                best_loss = avg_loss
                best_model_state = self.model.state_dict().copy()
            
            print(f"Epoch {epoch+1}/{num_epochs}:")
            print(f"  Total Loss: {avg_loss:.4f}")
            print(f"  Seq Loss: {total_seq_loss/batches_processed:.4f}")
            print(f"  Struct Loss: {total_struct_loss/batches_processed:.4f}")
            print(f"  Func Loss: {total_func_loss/batches_processed:.4f}")
            print(f"  VQ Loss: {total_vq_loss/batches_processed:.4f}")
            print("-" * 50)
        
        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
            print(f"Loaded best model with loss: {best_loss:.4f}")
        
        print("BIDIRECTIONAL MLM ESM3 Training Completed Successfully!")
        return dataset
    
    def mask_tokens(self, tokens, mask_prob=0.15):
        """Standard 15% masking like ESM3"""
        mask = torch.rand(tokens.shape) < mask_prob
        masked_tokens = tokens.clone()
        masked_tokens[mask] = 0  # Mask token
        return masked_tokens
    
    def get_embeddings(self, pdb_file):
        """Get meaningful bidirectional embeddings"""
        result = self.process_pdb_file(pdb_file)
        if result is None:
            return None
        
        with torch.no_grad():
            self.model.eval()
            _, _, _, embeddings = self.model(
                result['seq_tokens'].unsqueeze(0),
                result['struct_tokens'].unsqueeze(0),
                result['func_tokens'].unsqueeze(0)
            )
        
        return {
            'embeddings': embeddings.squeeze(0),
            'sequence': result['sequence'],
            'pdb_id': result['pdb_id']
        }
    
    def analyze_dataset(self, dataset):
        lengths = [len(item['seq_tokens']) for item in dataset]
        print(f"Dataset Analysis:")
        print(f"  Total structures: {len(dataset)}")
        print(f"  Min length: {min(lengths)}")
        print(f"  Max length: {max(lengths)}")
        print(f"  Avg length: {np.mean(lengths):.1f}")
        print(f"  Std length: {np.std(lengths):.1f}")

    def save_all_embeddings(self, pdb_folder, output_file):
        """Save embeddings for all proteins in the dataset"""
        print(f"Saving all embeddings to {output_file}")
        
        dataset = self.load_lc3_dataset(pdb_folder)
        all_embeddings = {}
        
        for item in dataset:
            pdb_id = item['pdb_id']
            try:
                embedding_result = self.get_embeddings(os.path.join(pdb_folder, f"{pdb_id}.pdb"))
                if embedding_result is not None:
                    all_embeddings[pdb_id] = {
                        'embeddings': embedding_result['embeddings'].cpu().numpy(),
                        'sequence': embedding_result['sequence']
                    }
                    print(f"Saved embeddings for {pdb_id} - shape: {all_embeddings[pdb_id]['embeddings'].shape}")
            except Exception as e:
                print(f"Error getting embeddings for {pdb_id}: {e}")
        
        # Save embeddings
        np.savez(output_file, **all_embeddings)
        
        # Save model
        model_file = output_file.replace('.npz', '_model.pth')
        torch.save(self.model.state_dict(), model_file)
        
        print(f"Saved {len(all_embeddings)} protein embeddings to {output_file}")
        print(f"Saved model to {model_file}")

if __name__ == "__main__":
    print("BIDIRECTIONAL MLM ESM3 - Fixed VQ Loss")
    
    working_esm3 = WorkingLC3ESM3(hidden_dim=256, max_sequence_length=500)
    
    pdb_folder = r"C:\Users\Swati_Sharma\Downloads\LC3\wildtype_dataset"
    trained_dataset = working_esm3.train_bidirectional_mlm(pdb_folder, num_epochs=10, batch_size=2)
    working_esm3.analyze_dataset(trained_dataset)
    
    print("BIDIRECTIONAL MLM ESM3 Training Completed Successfully!")

    # === FINAL LINES ===
    working_esm3.save_all_embeddings(pdb_folder, "LC3_all_proteins_embeddings.npz")
    print("ALL EMBEDDINGS + MODEL SAVED. YOU ARE DONE.")

ESM3 FOR LC3 PROTEINS - FIXED VQ LOSS
BIDIRECTIONAL MLM ESM3 - Fixed VQ Loss
Starting BIDIRECTIONAL MLM ESM3 Training
Using 15% masking + PROPER VQ loss
Loading LC3 PDB files from C:\Users\Swati_Sharma\Downloads\LC3\wildtype_dataset
Found 46 PDB files
Processed 1ugm - Length: 113
Processed 2k6q - Length: 121
Processed 2lue - Length: 119
Processed 2n9x - Length: 120
Processed 2ncn - Length: 128
Processed 2zjd - Length: 121
Processed 3eci - Length: 112
Processed 3vtu - Length: 122
Length adjustment: seq=60->60, coords=110->60 for 3vvw.pdb
Processed 3vvw - Length: 60
Processed 3wal - Length: 115
Processed 3wam - Length: 115
Processed 3wan - Length: 126
Processed 3wao - Length: 126
Processed 3wap - Length: 124
Processed 3x0w - Length: 124
Processed 4waa - Length: 126
Processed 4zdv - Length: 126
Processed 5cx3 - Length: 119
Processed 5d94 - Length: 119
Processed 5dcn - Length: 126
Processed 5dpr - Length: 124
Processed 5dpw - Length: 115
Processed 5gmv - Length: 120
Processed 5ms2 - Length

In [5]:
!pip install ESM

In [6]:
!pip install esm torch scikit-learn hdbscan umap-learn tqdm

In [12]:
from __future__ import annotations
import csv
import json
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import torch
from scipy import stats
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
import umap.umap_ as umap

In [14]:
import tqdm

In [19]:
!pip install esm  git+https://github.com/facebookresearch/esm
!pip install huggingface_hub
!pip install torch --index-url https://download.pytorch.org/whl/cu121   # (if using GPU)


  Cloning https://github.com/facebookresearch/esm to c:\users\swati_sharma\appdata\local\temp\pip-req-build-o7kbkb6k


  ERROR: Error [WinError 2] The system cannot find the file specified while executing command git version
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


4th December 2025 